In [1]:
from pathlib import Path
import json
import shutil

LAST_NAME = "Mejos"
STUDENT_ID = "TUPM-26-3605"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "user_behavior_dataset.csv"
export_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_device_users.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_device_summary.json"

if SEED_DIGIT % 2 == 0:
    target_os = "Android"
else:
    target_os = "iOS"
behavior_class = (SEED_DIGIT % 5) + 1

total_records = 0
os_records = 0
class_records = 0
invalid = 0
total_battery = 0.0
total_data = 0.0

with open(dataset_path, "r", encoding="utf-8") as file:
    next(file)

    with open(export_path, "w", encoding="utf-8") as output:
        output.write("User ID,Device Model,Operating System,Battery Drain (mAh/day),Data Usage (MB/day)\n")

        for line in file:
            total_records += 1
            fields = line.strip().split(",")

            if len(fields) == 11 and fields[5].isdigit() and fields[7].isdigit():
                if fields[2] == target_os:
                    os_records += 1

                    if fields[10] == f"{behavior_class}":
                        class_records += 1
                        total_battery += float(fields[5])
                        total_data += float(fields[7])
                        output.write(f"{fields[0]},{fields[1]},{fields[2]},{fields[5]},{fields[7]}\n")
            else:
                invalid += 1

# Averages (handles no matching records)
average_battery = total_battery / class_records if class_records else 0
average_data = total_data / class_records if class_records else 0

# Count verification: count the rows in the exported file
exported_rows = 0
with open(export_path, "r", encoding="utf-8") as file:
    next(file)
    for line in file:
        exported_rows += 1

verification = "MATCH" if exported_rows == class_records else "MISMATCH"

# JSON summary
summary = {
    "operating_system": target_os,
    "behavior_class": behavior_class,
    "total_records": total_records,
    "records_matching_os": os_records,
    "records_in_target_class": class_records,
    "average_battery_drain": round(average_battery, 2),
    "average_data_usage": round(average_data, 2),
    "exported_rows": exported_rows,
    "count_verification": verification
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# Backup of the exported CSV and its size
backup_path = BACKUP_DIR / export_path.name
shutil.copy2(export_path, backup_path)
backup_info = backup_path.stat()

print("Operating system and behavior class:", target_os, "| class", behavior_class)
print("Total records in dataset:", total_records)
print("Records matching operating system:", os_records)
print("Records in target behavior class:", class_records)
print("Average battery drain (mAh/day):", round(average_battery, 2))
print("Average data usage (MB/day):", round(average_data, 2))
print("Count verification:", exported_rows, "exported vs", class_records, "selected ->", verification)
print("Exported CSV file:", export_path)
print("JSON summary file:", json_path)
print("Backup file and size:", backup_path.name, "|", backup_info.st_size, "bytes")

print("\n=== FINAL OUTPUT ===")
print(f"{class_records} {target_os} users in behavior class {behavior_class} were exported.")
print(f"Average battery drain: {round(average_battery, 2)} mAh/day | Average data usage: {round(average_data, 2)} MB/day")

Operating system and behavior class: iOS | class 1
Total records in dataset: 700
Records matching operating system: 146
Records in target behavior class: 22
Average battery drain (mAh/day): 467.23
Average data usage (MB/day): 201.45
Count verification: 22 exported vs 22 selected -> MATCH
Exported CSV file: output\Mejos_TUPM-26-3605_device_users.csv
JSON summary file: output\Mejos_TUPM-26-3605_device_summary.json
Backup file and size: Mejos_TUPM-26-3605_device_users.csv | 674 bytes

=== FINAL OUTPUT ===
22 iOS users in behavior class 1 were exported.
Average battery drain: 467.23 mAh/day | Average data usage: 201.45 MB/day
